# U decomposition analysis

Generate Haar-random `SU(n)` matrices, decompose them with `unitary_to_G_rotations`, and convert the returned two-level rotations into ion-pulse parameters.

Notes:
- `mode="target"` converts the returned elimination rotations into the pulse sequence that synthesizes the target `U`.
- Each step returns `coupling`, `theta`, `phi`, and an extra diagonal phase `gamma`. `gamma` is the additional addressed-pair phase that must be tracked explicitly.
- For the `haar_su` sampler below, the residual `V` is numerically the identity. For a more general `U(n)` input, a residual diagonal can remain.
- Set `n = 29` when you want the active Ba-137 manifold size instead of a small test case.


In [2]:
import numpy as np
from numpy.linalg import norm

from U_decomp import unitary_to_G_rotations

np.set_printoptions(precision=4, suppress=True)


In [3]:
def haar_unitary(n, rng=None):
    rng = np.random.default_rng(rng)
    X = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(X)
    d = np.diag(R)
    D = d / np.abs(d)
    return Q * D


def haar_su(n, rng=None):
    U = haar_unitary(n, rng)
    phi = np.angle(np.linalg.det(U)) / n
    return U * np.exp(-1j * phi)


import numpy as np
from collections import deque
from math import gcd

# ============================================================
# Single-qudit Clifford group generator
# Supports:
#   - d = 2   (qubit)
#   - odd prime d = 3, 5, 7, ...
#
# It returns Clifford unitaries modulo global phase.
# ============================================================

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    p = 3
    while p * p <= n:
        if n % p == 0:
            return False
        p += 2
    return True


def modinv(a: int, m: int) -> int:
    """Modular inverse of a mod m."""
    a %= m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    raise ValueError(f"No modular inverse for {a} mod {m}")


def canonicalize_global_phase(U: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """
    Fix a canonical global phase so matrices that differ only by a global
    phase compare equal numerically.
    """
    U = np.array(U, dtype=complex)

    # Find first entry with non-negligible magnitude
    idx = None
    flat = U.flatten()
    for k, z in enumerate(flat):
        if abs(z) > tol:
            idx = k
            break

    if idx is None:
        raise ValueError("Zero matrix cannot be canonicalized.")

    phase = flat[idx] / abs(flat[idx])
    U = U / phase

    # Clean tiny numerical noise
    U.real[abs(U.real) < tol] = 0.0
    U.imag[abs(U.imag) < tol] = 0.0
    return U


def matrix_key(U: np.ndarray, decimals: int = 10) -> tuple:
    """
    Hashable key for a unitary modulo global phase.
    """
    Uc = canonicalize_global_phase(U)
    return tuple(np.round(Uc.flatten(), decimals=decimals))


def dft_matrix(d: int) -> np.ndarray:
    """Generalized Fourier transform F_d."""
    omega = np.exp(2j * np.pi / d)
    j, k = np.meshgrid(np.arange(d), np.arange(d), indexing="ij")
    return omega ** (j * k) / np.sqrt(d)


def qubit_generators():
    """
    Standard qubit generators H and S.
    These generate the single-qubit Clifford group modulo phase.
    """
    H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
    S = np.array([[1, 0], [0, 1j]], dtype=complex)
    return [H, S]


def odd_prime_qudit_generators(d: int):
    """
    Generators for single-qudit Clifford group in odd prime dimension d:
      F = Fourier transform
      P = quadratic phase gate with diagonal omega^{j(j-1)/2}
    """
    omega = np.exp(2j * np.pi / d)
    F = dft_matrix(d)

    inv2 = modinv(2, d)
    diag = [omega ** ((j * (j - 1) * inv2) % d) for j in range(d)]
    P = np.diag(diag)

    return [F, P]


def generalized_paulis(d: int):
    """
    Return X and Z for a single qudit of dimension d.
    Useful for verification.
    """
    omega = np.exp(2j * np.pi / d)

    X = np.zeros((d, d), dtype=complex)
    for j in range(d):
        X[(j + 1) % d, j] = 1

    Z = np.diag([omega ** j for j in range(d)])
    return X, Z


def generate_single_qudit_clifford_group(d: int, max_elements: int | None = None):
    """
    Generate the full single-qudit Clifford group modulo global phase.

    Parameters
    ----------
    d : int
        Dimension of the qudit.
        Supported: d = 2, or odd prime d.
    max_elements : int or None
        Optional safeguard to stop after reaching this many elements.

    Returns
    -------
    group : list[np.ndarray]
        List of unique Clifford matrices modulo global phase.
    """
    if d == 2:
        gens = qubit_generators()
    else:
        if not is_prime(d) or d % 2 == 0:
            raise ValueError(
                "This script supports d=2 or odd prime d only "
                "(e.g. 3, 5, 7, ...)."
            )
        gens = odd_prime_qudit_generators(d)

    # Include inverses too, so closure is reached faster
    gens = gens + [g.conj().T for g in gens]

    I = np.eye(d, dtype=complex)
    seen = {}
    q = deque()

    kI = matrix_key(I)
    seen[kI] = canonicalize_global_phase(I)
    q.append(seen[kI])

    while q:
        current = q.popleft()

        for g in gens:
            new = current @ g
            k = matrix_key(new)
            if k not in seen:
                seen[k] = canonicalize_global_phase(new)
                q.append(seen[k])

                if max_elements is not None and len(seen) >= max_elements:
                    return list(seen.values())

    return list(seen.values())


def verify_clifford(U: np.ndarray, d: int, tol: float = 1e-8) -> bool:
    """
    Basic verification: checks whether U X U† and U Z U† are generalized Pauli
    operators up to phase, by brute force over X^a Z^b for single-qudit case.
    """
    X, Z = generalized_paulis(d)

    paulis = []
    for a in range(d):
        Xa = np.linalg.matrix_power(X, a)
        for b in range(d):
            Zb = np.linalg.matrix_power(Z, b)
            paulis.append(Xa @ Zb)

    def matches_pauli(A):
        for P in paulis:
            # Compare up to global phase
            try:
                Ak = matrix_key(A)
                Pk = matrix_key(P)
                if Ak == Pk:
                    return True
            except Exception:
                pass
        return False

    Udag = U.conj().T
    return matches_pauli(U @ X @ Udag) and matches_pauli(U @ Z @ Udag)


if __name__ == "__main__":
    # Example 1: qubit
    d = 2
    G2 = generate_single_qudit_clifford_group(d)
    print(f"d={d}: generated {len(G2)} Cliffords modulo global phase")

    # Example 2: qutrit
    d = 3
    G3 = generate_single_qudit_clifford_group(d)
    print(f"d={d}: generated {len(G3)} Cliffords modulo global phase")

    # Verify a few elements
    print("Verification on first 5 qutrit elements:")
    for i, U in enumerate(G3[:5]):
        print(i, verify_clifford(U, 3))

    # Show one example matrix
    print("\nOne Clifford matrix for d=3:")
    print(G3[1])

d=2: generated 24 Cliffords modulo global phase
d=3: generated 216 Cliffords modulo global phase
Verification on first 5 qutrit elements:
0 True
1 True
2 True
3 True
4 True

One Clifford matrix for d=3:
[[ 0.5774+0.j   0.5774+0.j   0.5774+0.j ]
 [ 0.5774+0.j  -0.2887+0.5j -0.2887-0.5j]
 [ 0.5774+0.j  -0.2887-0.5j -0.2887+0.5j]]


In [4]:
def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def ion_pulse_unitary(coupling, theta, phi, dim):
    i, j = coupling
    U = np.eye(dim, dtype=complex)
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -1j * np.exp(1j * phi) * s
    U[j, i] = -1j * np.exp(-1j * phi) * s
    return U


def phase_update_unitary(coupling, gamma, dim):
    i, j = coupling
    D = np.eye(dim, dtype=complex)
    D[i, i] = np.exp(1j * gamma)
    D[j, j] = np.exp(-1j * gamma)
    return D


def rotation_matrix_to_pulse_parameters(G, tol=1e-10):
    G = np.asarray(G, dtype=complex)
    if G.ndim != 2 or G.shape[0] != G.shape[1]:
        raise ValueError("G must be square")

    dim = G.shape[0]
    candidates = [(p, q) for p in range(dim) for q in range(p + 1, dim)
                  if abs(G[p, q]) > tol or abs(G[q, p]) > tol]
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one coupled pair, found {candidates}")

    i, j = candidates[0]
    mask = np.ones_like(G, dtype=bool)
    mask[[i, j], :] = False
    mask[:, [i, j]] = False
    if norm(G[mask] - np.eye(dim, dtype=complex)[mask]) > 100 * tol:
        raise ValueError("Extra couplings detected outside the active 2x2 block")

    c = G[i, i]
    s = G[i, j]
    gamma = float(wrap_pi(np.angle(c)))
    theta = float(2.0 * np.arctan2(np.clip(np.abs(s), 0.0, 1.0), np.clip(np.abs(c), 0.0, 1.0)))
    phi = float((np.angle(s) - gamma + np.pi / 2) % (2 * np.pi))

    coupling = (i, j)
    reconstructed = phase_update_unitary(coupling, gamma, dim) @ ion_pulse_unitary(coupling, theta, phi, dim)

    return {
        "coupling": coupling,
        "theta": theta,
        "theta_over_pi": float(theta / np.pi),
        "phi": phi,
        "gamma": gamma,
        "reconstruction_error": float(norm(reconstructed - G)),
    }


def rotations_to_ion_schedule(rotation_mats, mode="target", tol=1e-10):
    if mode not in {"target", "elimination"}:
        raise ValueError("mode must be 'target' or 'elimination'")
    if len(rotation_mats) == 0:
        return []

    if mode == "target":
        sequence = [G.conj().T for G in rotation_mats[::-1]]
    else:
        sequence = list(rotation_mats)

    dim = sequence[0].shape[0]
    z_frame = np.zeros(dim, dtype=float)
    schedule = []

    for step_idx, G in enumerate(sequence, start=1):
        step = rotation_matrix_to_pulse_parameters(G, tol=tol)
        i, j = step["coupling"]
        gamma = step["gamma"]
        z_frame[i] += gamma
        z_frame[j] -= gamma

        step["step"] = step_idx
        step["z_phase_update"] = {i: gamma, j: -gamma}
        step["z_frame_after"] = z_frame.copy()
        schedule.append(step)

    return schedule


def schedule_to_lists(schedule):
    couplings = [step["coupling"] for step in schedule]
    thetas = [step["theta"] for step in schedule]
    phis = [step["phi"] for step in schedule]
    gammas = [step["gamma"] for step in schedule]
    return couplings, thetas, phis, gammas


def unitary_from_ion_schedule(couplings, thetas, phis, dim, gammas=None, return_z_frame=False):
    if not (len(couplings) == len(thetas) == len(phis)):
        raise ValueError("couplings, thetas, and phis must have the same length")
    if gammas is not None and len(gammas) != len(couplings):
        raise ValueError("gammas must match the number of pulses")

    U = np.eye(dim, dtype=complex)
    z_frame = np.zeros(dim, dtype=float)

    for idx, (coupling, theta, phi) in enumerate(zip(couplings, thetas, phis)):
        step_unitary = ion_pulse_unitary(coupling, theta, phi, dim)

        if gammas is not None:
            gamma = gammas[idx]
            step_unitary = phase_update_unitary(coupling, gamma, dim) @ step_unitary
            i, j = coupling
            z_frame[i] += gamma
            z_frame[j] -= gamma

        U = step_unitary @ U

    if return_z_frame:
        return U, z_frame
    return U

def qudit_qft(d: int, inverse: bool = False) -> np.ndarray:
    """
    Return the QFT matrix for a single qudit of dimension d.

    For inverse=False:
        F[j, k] = exp(2πi * j * k / d) / sqrt(d)

    For inverse=True:
        F†[j, k] = exp(-2πi * j * k / d) / sqrt(d)

    Args:
        d: qudit dimension (d >= 2)
        inverse: whether to return the inverse QFT

    Returns:
        A (d, d) complex numpy array.
    """
    if d < 2:
        raise ValueError("d must be at least 2")

    omega = np.exp((-2j if inverse else 2j) * np.pi / d)
    j = np.arange(d).reshape((d, 1))
    k = np.arange(d).reshape((1, d))

    return omega ** (j * k) / np.sqrt(d)

In [5]:
n = 16  # set to 29 for the active Ba-137 manifold
rng = 1234
center = 0

U = haar_su(n, rng=rng)

Hadamard = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Hadamard_2 = np.kron(Hadamard, Hadamard)
Hadamard_3 = np.kron(Hadamard_2, Hadamard)
U = Hadamard_3 
Hadamard_4 = np.kron(Hadamard_3, Hadamard)
U = Hadamard_4
n = 8
U = qudit_qft(d = n, inverse = False)
# U = generate_single_qudit_clifford_group(n, max_elements=10)[3]
# print(U@U.conj().T)
rotation_mats, V = unitary_to_G_rotations(U, center=center)

schedule = rotations_to_ion_schedule(rotation_mats, mode="target")
couplings, thetas, phis, gammas = schedule_to_lists(schedule)

print(f"n = {n}, center = {center}")
print(f"number of TAQR rotations: {len(rotation_mats)}")
print(f"||V - I|| = {norm(V - np.eye(n)):.3e}")
print("\nTarget-side pulse schedule:")

for step in schedule:
    i, j = step["coupling"]
    gamma = step["gamma"]
    if abs(gamma) < 1e-12:
        phase_note = "no extra diagonal phase"
    else:
        phase_note = f"extra phase: level {i} += {gamma:+.6f}, level {j} += {-gamma:+.6f}"

    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi={step['phi']:.6f} rad, gamma={gamma:+.6f} rad, {phase_note}"
    )

print("\nCopy/paste lists:")
print("couplings =", couplings)
print("thetas    =", thetas)
print("phis      =", phis)
print("gammas    =", gammas)


n = 8, center = 0
number of TAQR rotations: 28
||V - I|| = 1.414e+00

Target-side pulse schedule:
step  1: coupling=(0, 1), theta=1.155649 rad (0.367854 pi), phi=0.144246 rad, gamma=+1.646211 rad, extra phase: level 0 += +1.646211, level 1 += -1.646211
step  2: coupling=(0, 2), theta=1.296855 rad (0.412802 pi), phi=4.371617 rad, gamma=-2.800820 rad, extra phase: level 0 += -2.800820, level 2 += +2.800820
step  3: coupling=(0, 1), theta=2.611231 rad (0.831181 pi), phi=1.376398 rad, gamma=+1.258102 rad, extra phase: level 0 += +1.258102, level 1 += -1.258102
step  4: coupling=(0, 3), theta=1.912932 rad (0.608905 pi), phi=1.972102 rad, gamma=-0.401306 rad, extra phase: level 0 += -0.401306, level 3 += +0.401306
step  5: coupling=(0, 2), theta=0.900891 rad (0.286763 pi), phi=1.102041 rad, gamma=+0.000000 rad, no extra diagonal phase
step  6: coupling=(0, 1), theta=2.118403 rad (0.674309 pi), phi=3.201783 rad, gamma=-0.213077 rad, extra phase: level 0 += -0.213077, level 1 += +0.213077
step

In [6]:
U_pulses_only = unitary_from_ion_schedule(couplings, thetas, phis, dim=n)
U_with_gammas, z_frame = unitary_from_ion_schedule(
    couplings, thetas, phis, dim=n, gammas=gammas, return_z_frame=True
)

print("max per-step reconstruction error:", max(step['reconstruction_error'] for step in schedule))
print("pulse-only error vs target U:", norm(U_pulses_only - U))
print('Phase only U:')
print(U_pulses_only)
print("pulse+gamma error vs target U:", norm(U_with_gammas - U))
print('Pulse+gamma U:')
print(U_with_gammas)
print("final accumulated z-frame:")
print(z_frame)


max per-step reconstruction error: 4.660953740672398e-16
pulse-only error vs target U: 3.54018974866723
Phase only U:
[[ 0.0262-0.1726j -0.0928-0.0163j -0.3286-0.0983j  0.1226-0.221j
   0.2901-0.0273j  0.4812+0.2056j -0.5098+0.1884j  0.25  +0.25j  ]
 [ 0.3194-0.0444j  0.2178-0.1069j -0.1372-0.2743j -0.514 +0.3264j
  -0.1862+0.3343j -0.1525-0.0272j -0.147 +0.2353j  0.3536+0.j    ]
 [ 0.475 -0.0364j  0.0507+0.0252j  0.334 -0.061j  -0.0673-0.396j
   0.3955-0.053j  -0.0341+0.3116j  0.3255+0.0687j  0.25  -0.25j  ]
 [ 0.3589+0.5222j -0.2427+0.2398j -0.2649-0.1249j  0.0866+0.1243j
  -0.0253+0.0928j  0.3667-0.1889j  0.0203-0.2619j -0.    -0.3536j]
 [-0.3277+0.2281j -0.0033-0.0042j -0.0871-0.2741j -0.256 -0.1532j
   0.5765+0.1147j -0.3149-0.1807j -0.2511+0.0589j -0.25  -0.25j  ]
 [ 0.1013+0.1868j  0.434 -0.5214j  0.0678-0.255j   0.2289-0.1065j
  -0.0702+0.0645j  0.3185-0.1449j  0.1225+0.2995j -0.3536+0.j    ]
 [-0.1257+0.0183j  0.2208+0.2606j -0.247 +0.2673j -0.3452-0.1053j
   0.1329+0.4401j  0

## Gamma minimization and virtual-Z compilation

Allowing `theta` to run from `0` to `2*pi` gives a second exact branch for every step:

- original branch: `(theta, phi, gamma)`
- alternate branch: `(2*pi - theta, phi + pi, wrap_pi(gamma + pi))`

That alternate branch cannot eliminate `gamma` in general, but it can reduce `|gamma|` to at most `pi/2` for each step.

After that, the remaining `gamma` values can be folded into a running virtual-Z frame. With the convention used here, the exact compiled unitary is

`U_target = D_final @ U_pulses`

where `U_pulses` is built from the programmed LO phases and `D_final = diag(exp(1j * z_final))` is the final virtual-Z frame.


In [7]:
def minimize_gamma_for_step(step):
    alternate = dict(step)
    alternate["theta"] = float(2 * np.pi - step["theta"])
    alternate["theta_over_pi"] = float(alternate["theta"] / np.pi)
    alternate["phi"] = float((step["phi"] + np.pi) % (2 * np.pi))
    alternate["gamma"] = float(wrap_pi(step["gamma"] + np.pi))
    return alternate if abs(alternate["gamma"]) < abs(step["gamma"]) else dict(step)


def minimize_schedule_gamma(schedule):
    return [minimize_gamma_for_step(step) for step in schedule]


def virtual_z_diagonal(z_frame):
    return np.diag(np.exp(1j * np.asarray(z_frame, dtype=float)))


def compile_virtual_z_schedule(schedule):
    if len(schedule) == 0:
        return [], np.array([])

    if "z_frame_after" in schedule[0]:
        dim = len(schedule[0]["z_frame_after"])
    else:
        dim = max(max(step["coupling"]) for step in schedule) + 1

    frame = np.zeros(dim, dtype=float)
    compiled = []

    for step in schedule:
        i, j = step["coupling"]
        programmed_phi = float((step["phi"] - (frame[i] - frame[j])) % (2 * np.pi))

        compiled_step = dict(step)
        compiled_step["frame_before"] = frame.copy()
        compiled_step["phi_programmed"] = programmed_phi

        frame[i] += step["gamma"]
        frame[j] -= step["gamma"]

        compiled_step["frame_after"] = frame.copy()
        compiled.append(compiled_step)

    return compiled, frame


def programmed_schedule_to_unitary(couplings, thetas, programmed_phis, dim, final_frame=None):
    U_pulses = unitary_from_ion_schedule(couplings, thetas, programmed_phis, dim)
    if final_frame is None:
        return U_pulses
    return virtual_z_diagonal(final_frame) @ U_pulses


In [8]:
schedule_min = minimize_schedule_gamma(schedule)
compiled_schedule, z_final = compile_virtual_z_schedule(schedule_min)

couplings_vz = [step["coupling"] for step in compiled_schedule]
thetas_vz = [step["theta"] for step in compiled_schedule]
phis_vz = [step["phi_programmed"] for step in compiled_schedule]
gammas_vz = [step["gamma"] for step in compiled_schedule]

print("max |gamma| before minimization:", max(abs(step["gamma"]) for step in schedule))
print("max |gamma| after minimization :", max(abs(step["gamma"]) for step in schedule_min))
print("\nVirtual-Z compiled schedule:")

couplings = []
thetas = []
phases = []

for step in compiled_schedule:
    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi_programmed={step['phi_programmed']:.6f} rad, gamma={step['gamma']:+.6f} rad"
    )
    couplings.append(step["coupling"])
    thetas.append(step["theta"])
    phases.append(step["phi_programmed"])

print("\nProgrammed LO phases:", phis_vz)
print("Final virtual-Z frame:")
print(z_final)

U_programmed_only = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n)
U_programmed_exact = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n, final_frame=z_final)

print("\nprogrammed-pulses-only error vs target U:", norm(U_programmed_only - U))
# print(U_programmed_only)
print("programmed-pulses + final virtual-Z error vs target U:", norm(U_programmed_exact - U))
print(U_programmed_exact)
print(U)


max |gamma| before minimization: 3.0501014209884056
max |gamma| after minimization : 1.5707963267948948

Virtual-Z compiled schedule:
step  1: coupling=(0, 1), theta=5.127537 rad (1.632146 pi), phi_programmed=3.285838 rad, gamma=-1.495382 rad
step  2: coupling=(0, 2), theta=4.986330 rad (1.587198 pi), phi_programmed=2.725406 rad, gamma=+0.340772 rad
step  3: coupling=(0, 1), theta=2.611231 rad (0.831181 pi), phi_programmed=4.026389 rad, gamma=+1.258102 rad
step  4: coupling=(0, 3), theta=1.912932 rad (0.608905 pi), phi_programmed=1.868609 rad, gamma=-0.401306 rad
step  5: coupling=(0, 2), theta=0.900891 rad (0.286763 pi), phi_programmed=1.059082 rad, gamma=+0.000000 rad
step  6: coupling=(0, 1), theta=2.118403 rad (0.674309 pi), phi_programmed=3.736875 rad, gamma=-0.213077 rad
step  7: coupling=(0, 4), theta=2.116181 rad (0.673601 pi), phi_programmed=1.608583 rad, gamma=+0.473104 rad
step  8: coupling=(0, 3), theta=0.985945 rad (0.313836 pi), phi_programmed=2.819265 rad, gamma=+0.00000

In [9]:
print("couplings = ", couplings)
print("theta = ", thetas)
print("phases = ", phases)

couplings =  [(0, 1), (0, 2), (0, 1), (0, 3), (0, 2), (0, 1), (0, 4), (0, 3), (0, 2), (0, 1), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1), (0, 7), (0, 6), (0, 5), (0, 4), (0, 3), (0, 2), (0, 1)]
theta =  [5.1275365674439355, 4.986330032277465, 2.6112312120283705, 1.9129317204931977, 0.9008912870641983, 2.1184028416966045, 2.116181127725508, 0.9859447144604152, 1.0538130304171098, 1.7748399714993186, 2.2092013790709886, 1.0100669832973168, 1.1302245290232853, 1.3366116235698977, 1.214488634668413, 4.013385238686697, 1.0306750782963638, 1.1803474944396672, 1.286293984052291, 1.2606488666583338, 5.345926895956892, 2.418858405776376, 0.7751933733103633, 0.8410686705679322, 0.9272952180016141, 1.0471975511965992, 1.2309594173407756, 1.5707963267948974]
phases =  [3.2858382536515087, 2.725405521769697, 4.026388789474553, 1.8686094229620545, 1.0590817940669779, 3.736875332078083, 1.6085825433925551, 2.8192651625607636, 5.053857436147247, 3.4286530701

In [10]:
print(sum(thetas)/np.pi)

17.00326810822422


In [15]:
theta =  [1.5, 1, 1., 1.5, 2.0 * np.arcsin(np.sqrt(1/3))/np.pi, 2/3, 1 + 2.0 * np.arcsin(np.sqrt(1/3))/np.pi, 1, 1, 0.5, 0.5, 0.5, 1.0, 1.5, 1.0, 1.5, 1, 1.5, 1.5, 1, 1.5]
print(sum(theta))

22.45031977072788


## Closing the final frame physically

Branch choices alone do **not** give arbitrary control over the final diagonal. On a star centered at level `0`, they only toggle `pi` parities on the addressed edge.

That means branch-only closure is possible only if every relative phase `z_j - z_0` is already `0` or `pi` modulo `2*pi`.

When that condition fails, the correct fix is to append an explicit end correction for

`D_final = diag(exp(1j * z_final))`

For a star topology, use

`D_final = prod_{j>0} Z_(0,j)(-z_j)`

and implement each `Z_(0,j)(alpha)` with two resonant `pi` pulses on edge `(0, j)` whose phase difference is `alpha - pi`.


In [11]:
def is_global_phase_only(z_frame, tol=1e-9):
    z = np.asarray(z_frame, dtype=float)
    return all(abs(wrap_pi(value - z[0])) < tol for value in z)


def branch_only_can_close_frame(z_frame, center=0, tol=1e-9):
    z = np.asarray(z_frame, dtype=float)
    relative_phases = []
    for j in range(len(z)):
        if j == center:
            continue
        delta = float(wrap_pi(z[j] - z[center]))
        relative_phases.append((j, delta))

    closable = all(abs(delta) < tol or abs(abs(delta) - np.pi) < tol for _, delta in relative_phases)
    return closable, relative_phases


def z_gate_from_two_pi_pulses(coupling, alpha, phi_base=0.0):
    phi1 = float(phi_base % (2 * np.pi))
    phi2 = float((phi_base + alpha - np.pi) % (2 * np.pi))
    return [
        {"coupling": coupling, "theta": float(np.pi), "phi_programmed": phi1},
        {"coupling": coupling, "theta": float(np.pi), "phi_programmed": phi2},
    ]


def compile_final_frame_correction(z_frame, center=0, phi_base=0.0, tol=1e-9):
    z = np.asarray(z_frame, dtype=float)
    if abs(wrap_pi(np.sum(z))) > 1e-6:
        raise ValueError("z_frame must sum to 0 modulo 2*pi for an SU(d) correction")

    correction_angles = []
    correction_steps = []

    for j in range(len(z)):
        if j == center:
            continue
        alpha = float(wrap_pi(-z[j]))
        correction_angles.append((center, j, alpha))
        if abs(alpha) < tol:
            continue
        correction_steps.extend(z_gate_from_two_pi_pulses((center, j), alpha, phi_base=phi_base))

    return correction_angles, correction_steps


def correction_unitary_from_steps(correction_steps, dim):
    if len(correction_steps) == 0:
        return np.eye(dim, dtype=complex)
    couplings_corr = [step["coupling"] for step in correction_steps]
    thetas_corr = [step["theta"] for step in correction_steps]
    phis_corr = [step["phi_programmed"] for step in correction_steps]
    return unitary_from_ion_schedule(couplings_corr, thetas_corr, phis_corr, dim)


In [13]:
closable, relative_phases = branch_only_can_close_frame(z_final, center=0)
print("Global-phase only already:", is_global_phase_only(z_final))
print("Branch-only closure possible:", closable)
print("Relative phases versus level 0:")
for j, delta in relative_phases:
    print(f"  level {j}: {delta:+.6f} rad")

correction_angles, correction_steps = compile_final_frame_correction(z_final, center=0, phi_base=0.0)
print("\nRequired end-correction edge phases:")
for center_idx, j, alpha in correction_angles:
    print(f"  Z_({center_idx},{j})({alpha:+.6f} rad)")

print(f"\nNumber of extra pulses to close the frame: {len(correction_steps)}")
for idx, step in enumerate(correction_steps, start=1):
    print(
        f"corr {idx:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad, phi_programmed={step['phi_programmed']:.6f} rad"
    )

U_correction = correction_unitary_from_steps(correction_steps, n)
U_closed = U_correction @ U_programmed_only
print(U_closed)
print(U)
print("\nCorrection unitary vs D_final:", norm(U_correction - virtual_z_diagonal(z_final)))
print("Error after explicit physical frame closure:", norm(U_closed - U))


Global-phase only already: False
Branch-only closure possible: False
Relative phases versus level 0:
  level 1: +2.232519 rad
  level 2: -2.019692 rad
  level 3: -1.277613 rad
  level 4: -2.152024 rad
  level 5: -1.267428 rad
  level 6: -1.770410 rad
  level 7: -0.893521 rad

Required end-correction edge phases:
  Z_(0,1)(+2.371747 rad)
  Z_(0,2)(+0.340772 rad)
  Z_(0,3)(-0.401306 rad)
  Z_(0,4)(+0.473104 rad)
  Z_(0,5)(-0.411492 rad)
  Z_(0,6)(+0.091491 rad)
  Z_(0,7)(-0.785398 rad)

Number of extra pulses to close the frame: 14
corr  1: coupling=(0, 1), theta=3.141593 rad, phi_programmed=0.000000 rad
corr  2: coupling=(0, 1), theta=3.141593 rad, phi_programmed=5.513340 rad
corr  3: coupling=(0, 2), theta=3.141593 rad, phi_programmed=0.000000 rad
corr  4: coupling=(0, 2), theta=3.141593 rad, phi_programmed=3.482365 rad
corr  5: coupling=(0, 3), theta=3.141593 rad, phi_programmed=0.000000 rad
corr  6: coupling=(0, 3), theta=3.141593 rad, phi_programmed=2.740287 rad
corr  7: coupling=(0